# Aula 07 — Camadas lineares, ativações e inicialização

Objetivo: transportar a convenção `X @ W + b` do M5 para `nn.Linear`, conferir forward/backward e auditar a inicialização antes do treinamento.

Requer Python >=3.10, PyTorch >=2.6, NumPy >=1.24 e Matplotlib >=3.6. Ambiente validado em 9 de setembro de 2026: Python 3.12.14, PyTorch 2.6.0+cpu, NumPy 2.3.5 e Matplotlib 3.10.8, CPU. `nbformat >=5.10` serve à validação do arquivo. Reinicie e execute todas as células em ordem. Nenhum download de dados, credencial ou GPU é necessário.

A fixture sintética usa seed 20260907; não há treino, ajuste de estatísticas, seleção de hiperparâmetros nem medida de generalização. Uma figura SVG será criada em `assets/` no diretório de execução. Outputs são limpos no commit; resultados confirmados estão na aula.

In [ ]:
import math
import sys
from pathlib import Path
import numpy as np
import torch
from torch import nn
import torch.nn.functional as F
import matplotlib
import matplotlib.pyplot as plt

SEED = 20260907
torch.manual_seed(SEED)
torch.set_num_threads(1)
rng = np.random.default_rng(SEED)
dtype = torch.float64
checks = []
def check(name, condition):
    assert bool(condition), name
    checks.append(name)
def generator(seed=SEED):
    return torch.Generator(device='cpu').manual_seed(seed)
print('Python', sys.version.split()[0], '| torch', torch.__version__, '| NumPy', np.__version__, '| Matplotlib', matplotlib.__version__)
check('CPU', torch.empty(0).device.type == 'cpu')

## 1. Uma transposição que muda o significado

A camada manual usa peso `(D,H)`; Linear armazena `(H,D)` e calcula `X @ weight.T + bias`. Copiamos valores com `copy_` em `no_grad`, preservando o Parameter. O exemplo quadrado é intencional: um erro de orientação continua executável.

In [ ]:
x_small = torch.tensor([[1., 2.]], dtype=dtype)
w_small = torch.tensor([[1., 2.], [3., 4.]], dtype=dtype)
b_small = torch.tensor([.5, -.5], dtype=dtype)
layer = nn.Linear(2, 2, dtype=dtype)
with torch.no_grad():
    layer.weight.copy_(w_small.T)
    layer.bias.copy_(b_small)
expected = x_small @ w_small + b_small
check('resultado manual', torch.equal(expected, torch.tensor([[7.5, 9.5]], dtype=dtype)))
check('Linear com peso transposto', torch.equal(layer(x_small), expected))
wrong = F.linear(x_small, w_small, b_small)
orientation_error = (wrong - expected).abs().max().item()
check('orientação errada detectada', orientation_error == 2)
print('Correto:', expected.tolist(), '| sem transposição:', wrong.tolist(), '| erro máximo:', orientation_error)

## 2. Contrato do último eixo

Linear(3,4) transforma o último eixo de comprimento 3 em 4. Os eixos anteriores são preservados; isso não mistura exemplos ou posições. `bias=False` remove o parâmetro aditivo.

In [ ]:
rect = nn.Linear(3, 4, dtype=dtype)
x3 = torch.tensor(rng.normal(size=(2, 5, 3)), dtype=dtype)
check('peso retangular', rect.weight.shape == (4, 3))
check('shape com eixos extras', rect(x3).shape == (2, 5, 4))
check('último eixo equivale ao laço', torch.allclose(rect(x3), torch.stack([torch.stack([rect(x3[i,j]) for j in range(5)]) for i in range(2)]), atol=1e-14, rtol=0))
no_bias = nn.Linear(3, 4, bias=False, dtype=dtype)
check('bias ausente', no_bias.bias is None)
check('12 escalares sem bias', sum(p.numel() for p in no_bias.parameters()) == 12)
print('Entrada:', tuple(x3.shape), '| saída:', tuple(rect(x3).shape))

## 3. Fixture MLP e referência NumPy

Usamos B=5, D=3, H=4, C=2 e metade do MSE sobre os dez resíduos. Os geradores não são equivalentes entre bibliotecas: os valores são gerados uma vez em NumPy e copiados. O backward abaixo é independente do autograd.

In [ ]:
X_np = rng.normal(size=(5, 3))
T_np = rng.normal(size=(5, 2))
p_np = {'W1': rng.normal(scale=.2, size=(3,4)), 'b1': rng.normal(scale=.1, size=4),
        'W2': rng.normal(scale=.2, size=(4,2)), 'b2': rng.normal(scale=.1, size=2)}
A_np = np.tanh(X_np @ p_np['W1'] + p_np['b1'])
S_np = A_np @ p_np['W2'] + p_np['b2']
loss_np = float(np.mean((S_np - T_np)**2) / 2)
dS = (S_np - T_np) / T_np.size
dZ = (dS @ p_np['W2'].T) * (1 - A_np**2)
g_np = {'W1': X_np.T @ dZ, 'b1': dZ.sum(axis=0), 'W2': A_np.T @ dS,
        'b2': dS.sum(axis=0), 'X': dZ @ p_np['W1'].T}
X = torch.tensor(X_np, dtype=dtype, requires_grad=True)
T = torch.tensor(T_np, dtype=dtype)
check('26 parâmetros', sum(v.size for v in p_np.values()) == 26)
print('Loss NumPy:', f'{loss_np:.12f}')

## 4. Linear + Tanh + Linear com os mesmos pesos

O construtor sorteia valores, mas todos os quatro parâmetros são sobrescritos antes da comparação. Não aplique outra inicialização depois da cópia. Gradientes das matrizes devem voltar à orientação do M5 antes de comparar.

In [ ]:
net = nn.Sequential(nn.Linear(3, 4, dtype=dtype), nn.Tanh(), nn.Linear(4, 2, dtype=dtype))
with torch.no_grad():
    net[0].weight.copy_(torch.tensor(p_np['W1'].T, dtype=dtype))
    net[0].bias.copy_(torch.tensor(p_np['b1'], dtype=dtype))
    net[2].weight.copy_(torch.tensor(p_np['W2'].T, dtype=dtype))
    net[2].bias.copy_(torch.tensor(p_np['b2'], dtype=dtype))
S = net(X)
loss = ((S - T)**2).mean() / 2
loss.backward()
g_torch = {'W1': net[0].weight.grad.T.numpy(), 'b1': net[0].bias.grad.numpy(),
           'W2': net[2].weight.grad.T.numpy(), 'b2': net[2].bias.grad.numpy(), 'X': X.grad.numpy()}
forward_error = float(np.max(np.abs(S.detach().numpy() - S_np)))
gradient_errors = {k:float(np.max(np.abs(g_torch[k]-g_np[k]))) for k in g_np}
check('forward NumPy versus Linear', forward_error < 1e-14)
check('mesma loss', abs(loss.item()-loss_np) < 1e-14)
for k, err in gradient_errors.items(): check('gradiente '+k, err < 1e-14)
check('26 escalares no Sequential', sum(p.numel() for p in net.parameters()) == 26)
print('Erro forward:', forward_error, '| erros dos gradientes:', gradient_errors)

## 5. Duas camadas afins sem ativação colapsam em uma

Na convenção M5, W_equiv=W1@W2 e b_equiv=b1@W2+b2. Tanh introduz uma função diferente; a diferença numérica na fixture não é uma prova de superioridade preditiva.

In [ ]:
w_equiv = p_np['W1'] @ p_np['W2']
b_equiv = p_np['b1'] @ p_np['W2'] + p_np['b2']
composed = (X_np @ p_np['W1'] + p_np['b1']) @ p_np['W2'] + p_np['b2']
collapsed = X_np @ w_equiv + b_equiv
collapse_error = float(np.max(np.abs(composed-collapsed)))
nonlinear_gap = float(np.max(np.abs(S_np-collapsed)))
check('composição afim colapsa', collapse_error < 1e-14)
check('tanh muda a função nesta fixture', nonlinear_gap > 1e-5)
print('Erro do colapso afim:', collapse_error, '| diferença com tanh:', nonlinear_gap)

## 6. Ativações: módulos, funções e derivadas

Verificamos valores e derivadas fora das quinas. Para ReLU no zero, fazemos uma verificação separada da convenção do PyTorch. Usamos inplace=False para conservar a entrada nos ramos do experimento.

In [ ]:
z_values = torch.tensor([-3., -1., 0., 1., 3.], dtype=dtype)
alpha = .1
activations = {
    'sigmoid': (nn.Sigmoid(), torch.sigmoid, lambda z: torch.sigmoid(z)*(1-torch.sigmoid(z))),
    'tanh': (nn.Tanh(), torch.tanh, lambda z: 1-torch.tanh(z)**2),
    'relu': (nn.ReLU(inplace=False), F.relu, lambda z: (z>0).to(dtype)),
    'leaky_relu': (nn.LeakyReLU(alpha, inplace=False), lambda z: F.leaky_relu(z, alpha), lambda z: torch.where(z>0, torch.ones_like(z), torch.full_like(z, alpha))),
}
for name,(module,fn,derivative) in activations.items():
    z = z_values.clone().requires_grad_()
    a = module(z)
    dz, = torch.autograd.grad(a.sum(), z)
    check(name+' módulo versus função', torch.allclose(a, fn(z), atol=1e-14, rtol=0))
    check(name+' derivada fora da quina', torch.allclose(dz[z!=0], derivative(z)[z!=0], atol=1e-14, rtol=0))
    check(name+' sem parâmetros', len(list(module.parameters())) == 0)
    check(name+' entrada preservada', torch.equal(z.detach(), z_values))
    print(name, 'saída:', a.detach().tolist())
z0 = torch.tensor([0.], dtype=dtype, requires_grad=True)
relu_zero, = torch.autograd.grad(nn.ReLU()(z0).sum(), z0)
check('ReLU usa derivada zero na quina', relu_zero.item() == 0)
sigmoid_slope = float((torch.sigmoid(torch.tensor(10.,dtype=dtype))*(1-torch.sigmoid(torch.tensor(10.,dtype=dtype)))).item())
check('sigmoid saturada tem pequena derivada', sigmoid_slope < 5e-5)
print('Derivada sigmoid em 10:', sigmoid_slope)

## 7. Gráfico das funções e derivadas

Texto alternativo: sigmoid e tanh saturam nos extremos; ReLU zera valores negativos e tem inclinação um à direita; Leaky ReLU mantém inclinação 0,1 à esquerda. O segundo painel exclui o ponto zero das curvas com quina. As escalas verticais dos painéis são diferentes.

A figura é gerada dos valores das funções; `assets/07-ativacoes-derivadas.svg` é o arquivo reproduzível no diretório de execução.

In [ ]:
grid = torch.linspace(-5, 5, 501, dtype=dtype)
fig, axes = plt.subplots(1, 2, figsize=(11, 4), constrained_layout=True)
colors = ['#0072B2', '#D55E00', '#009E73', '#CC79A7']
styles = ['-', '--', '-.', ':']
for (name,(module,fn,derivative)), color, style in zip(activations.items(), colors, styles):
    axes[0].plot(grid.numpy(), fn(grid).numpy(), label=name, color=color, linestyle=style)
    deriv = derivative(grid).numpy().copy()
    if name in ('relu','leaky_relu'): deriv[np.abs(grid.numpy())<1e-12] = np.nan
    axes[1].plot(grid.numpy(), deriv, label=name, color=color, linestyle=style)
for ax,title,ylabel in zip(axes, ['Ativações', 'Derivadas fora das quinas'], ['phi(z)', "phi'(z)"]):
    ax.set(xlabel='z', ylabel=ylabel, title=title)
    ax.grid(alpha=.25)
    ax.legend(fontsize=8)
assets = Path('assets')
assets.mkdir(exist_ok=True)
figure_path = assets/'07-ativacoes-derivadas.svg'
with plt.rc_context({'svg.hashsalt':'m6-a07', 'svg.fonttype':'none'}):
    fig.savefig(figure_path, metadata={'Date':None, 'Title':'Ativações e derivadas — M6 Aula 07', 'Description':'Sigmoid e tanh saturam; ReLU tem derivada zero à esquerda; Leaky ReLU conserva inclinação 0,1. Quinas não representadas como derivadas clássicas.'})
plt.show()
plt.close(fig)
check('figura criada', figure_path.is_file())
print('Figura:', figure_path)

## 8. Ganhos e escalas declaradas

Xavier com gain=1 reproduz a variância-alvo básica do M5. calculate_gain('tanh') retorna 5/3: não é o mesmo protocolo. Kaiming para Leaky ReLU usa a mesma inclinação negativa configurada no forward.

In [ ]:
g_tanh = nn.init.calculate_gain('tanh')
g_relu = nn.init.calculate_gain('relu')
g_leaky = nn.init.calculate_gain('leaky_relu', alpha)
check('ganho tanh', g_tanh == 5/3)
check('ganho ReLU', abs(g_relu-math.sqrt(2)) < 1e-14)
check('ganho Leaky ReLU', abs(g_leaky-math.sqrt(2/(1+alpha**2))) < 1e-14)
fan_in, fan_out = 128, 64
scales = {'default_std':1/math.sqrt(3*fan_in),
          'xavier_g1_std':math.sqrt(2/(fan_in+fan_out)),
          'xavier_tanh_std':g_tanh*math.sqrt(2/(fan_in+fan_out)),
          'he_relu_std':math.sqrt(2/fan_in)}
check('He variância seis vezes default', abs(scales['he_relu_std']**2/scales['default_std']**2-6) < 1e-12)
print('Desvios-padrão teóricos, fan_in=128, fan_out=64:', scales)

## 9. Auditar a inicialização padrão de Linear

Para esta versão, peso e bias são uniformes em [-1/sqrt(fan_in), 1/sqrt(fan_in)]. O nome Kaiming no código interno não significa ganho ReLU: o argumento a=sqrt(5) produz ganho 1/sqrt(3). Validamos limites e variância de uma matriz grande, sem exigir média ou variância amostral exatas.

In [ ]:
with torch.random.fork_rng(devices=[]):
    torch.manual_seed(SEED+1)
    default_layer = nn.Linear(1024, 256, dtype=dtype)
bound = 1/math.sqrt(1024)
default_var = default_layer.weight.detach().var(unbiased=False).item()
check('peso default nos limites', default_layer.weight.detach().abs().max().item() <= bound)
check('bias default nos limites', default_layer.bias.detach().abs().max().item() <= bound)
check('variância default aproximada', abs(default_var/(1/(3*1024))-1) < .02)
check('bias default não é todo zero', torch.count_nonzero(default_layer.bias).item() > 0)
check('ganho com a sqrt5', abs(nn.init.calculate_gain('leaky_relu', math.sqrt(5))-1/math.sqrt(3)) < 1e-14)
print('Variância default observada:', default_var, '| alvo:', 1/(3*1024))

## 10. Xavier com e sem ganho tanh

Duas matrizes usam a mesma sequência normal e diferem somente no ganho. Isso isola uma alteração metodológica: multiplicar o ganho por 5/3 multiplica a variância por 25/9. Esta comparação não escolhe o melhor ganho para uma tarefa.

In [ ]:
xavier1 = torch.empty(256, 1024, dtype=dtype)
xavier_tanh = torch.empty_like(xavier1)
nn.init.xavier_normal_(xavier1, gain=1., generator=generator(SEED+2))
nn.init.xavier_normal_(xavier_tanh, gain=g_tanh, generator=generator(SEED+2))
xavier_ratio = (xavier_tanh.var(unbiased=False)/xavier1.var(unbiased=False)).item()
check('ganho multiplica amostra', torch.allclose(xavier_tanh, xavier1*g_tanh, atol=1e-14, rtol=0))
check('ganho multiplica variância por quadrado', abs(xavier_ratio-25/9) < 1e-12)
check('variância Xavier g1', abs(xavier1.var(unbiased=False).item()/(2/1280)-1) < .02)
print('Razão de variâncias tanh / ganho 1:', xavier_ratio)

## 11. Contraprova de fan_in: matriz manual inicializada na orientação errada

W_manual tem shape (1024,256) e será usado como X@W_manual. A API Kaiming infere fan_in do segundo eixo; por isso deve receber W_manual.T. A chamada errada usa 256 em vez de 1024 e quadruplica a variância-alvo. Medimos o segundo momento após ReLU no mesmo lote sintético.

In [ ]:
correct_manual = torch.empty(1024, 256, dtype=dtype)
wrong_manual = torch.empty_like(correct_manual)
nn.init.kaiming_normal_(correct_manual.T, mode='fan_in', nonlinearity='relu', generator=generator(SEED+3))
nn.init.kaiming_normal_(wrong_manual, mode='fan_in', nonlinearity='relu', generator=generator(SEED+3))
correct_var = correct_manual.var(unbiased=False).item()
wrong_var = wrong_manual.var(unbiased=False).item()
fan_ratio = wrong_var/correct_var
check('variância correta He', abs(correct_var/(2/1024)-1) < .02)
check('variância errada He', abs(wrong_var/(2/256)-1) < .02)
check('razão de variância próxima de quatro', abs(fan_ratio-4) < .1)
audit_x = torch.randn(128, 1024, dtype=dtype, generator=generator(SEED+4))
moments = {'correct': torch.relu(audit_x @ correct_manual).square().mean().item(),
           'wrong': torch.relu(audit_x @ wrong_manual).square().mean().item()}
check('momentos finitos', all(math.isfinite(v) for v in moments.values()))
check('escala errada amplifica neste ensaio', moments['wrong']/moments['correct'] > 3)
print('Variâncias correta / errada:', correct_var, wrong_var, '| razão:', fan_ratio)
print('Segundo momento após ReLU:', moments)

## 12. Inicialização explícita de uma MLP nova

Mantemos o modelo de paridade intacto. A política abaixo aplica He/ReLU ao corpo, Xavier ganho 1 à saída afim e zero aos biases. O gerador é criado uma vez por rede, não reiniciado em cada camada. A rede não é treinada aqui.

In [ ]:
def fresh_net(seed):
    with torch.random.fork_rng(devices=[]):
        torch.manual_seed(seed)
        m = nn.Sequential(nn.Linear(3, 4, dtype=dtype), nn.ReLU(), nn.Linear(4, 2, dtype=dtype))
    g = generator(seed)
    nn.init.kaiming_normal_(m[0].weight, mode='fan_in', nonlinearity='relu', generator=g)
    nn.init.zeros_(m[0].bias)
    nn.init.xavier_uniform_(m[2].weight, gain=1., generator=g)
    nn.init.zeros_(m[2].bias)
    return m
fresh1, fresh2, fresh3 = fresh_net(SEED+5), fresh_net(SEED+5), fresh_net(SEED+6)
check('mesma configuração e seed reproduz pesos', all(torch.equal(a,b) for a,b in zip(fresh1.parameters(),fresh2.parameters())))
check('outra seed muda pesos', not torch.equal(fresh1[0].weight, fresh3[0].weight))
check('biases zerados', all(torch.count_nonzero(m.bias).item()==0 for m in [fresh1[0],fresh1[2]]))
check('inicialização mantém leaf e requires_grad', all(p.is_leaf and p.requires_grad for p in fresh1.parameters()))
check('inicialização sem gradiente acumulado', all(p.grad is None for p in fresh1.parameters()))
check('rede nova finita', torch.isfinite(fresh1(X.detach())).all())
check('fixture de paridade não reinicializada', torch.equal(net[0].weight.detach(), torch.tensor(p_np['W1'].T,dtype=dtype)))
print('Rede nova reproduzível; fixture de paridade preservada.')

## 13. Auditoria final

Separar equivalência de implementação de escolha de arquitetura evita chamar uma mudança de ativação ou de distribuição de mero detalhe técnico. As tolerâncias estatísticas são largas para amostras finitas; limites do gerador e identidade das operações não são confundidos com garantias de generalização.

In [ ]:
check('nomes das verificações únicos', len(checks) == len(set(checks)))
check('todos os gradientes finitos', all(np.isfinite(g).all() for g in g_torch.values()))
print(f'{len(checks)}/{len(checks)} verificações aprovadas')
print('Loss:', f'{loss_np:.12f}', '| maior erro de gradiente:', max(gradient_errors.values()))
print('Razão de variâncias por erro de fan:', fan_ratio)
print('Não foram executados treinamento, GPU ou teste de generalização.')

## Exercícios e respostas

1. Linear(7,3) tem qual shape de peso? (3,7); são 21 pesos e, com bias, mais 3 números.
2. É suficiente testar a cópia em uma matriz quadrada? Não: a transposição errada mantém shapes válidos. Compare valores e use também caso retangular.
3. Como comparar weight.grad com o gradiente manual? Transponha-o para a orientação (entrada,saída).
4. Trocar tanh por ReLU preserva a função? Em geral, não. É uma ablação de arquitetura, não uma equivalência de implementação.
5. Por que Xavier tanh difere do M5 básico? O ganho 5/3 altera a escala; o protocolo básico usa ganho 1.
6. Por que a variância de He foi quatro vezes maior na contraprova? fan_in foi inferido como 256 em vez de 1024.
7. Inicializar todas as camadas com a mesma seed local é independência? Pode repetir matrizes de mesmo shape; consuma o fluxo de um gerador por rede.
8. ReLU possui derivada clássica no zero? Não. A verificação testa a convenção operacional de autograd, não diferenciabilidade matemática.

## Fontes e continuação

Documentação oficial PyTorch 2.6, verificada em 9 de setembro de 2026:

- [Linear](https://docs.pytorch.org/docs/2.6/generated/torch.nn.Linear.html)
- [Inicialização](https://docs.pytorch.org/docs/2.6/nn.init.html)
- [ReLU](https://docs.pytorch.org/docs/2.6/generated/torch.nn.ReLU.html)
- [LeakyReLU](https://docs.pytorch.org/docs/2.6/generated/torch.nn.LeakyReLU.html)
- [Sigmoid](https://docs.pytorch.org/docs/2.6/generated/torch.nn.Sigmoid.html)
- [Tanh](https://docs.pytorch.org/docs/2.6/generated/torch.nn.Tanh.html)

Fundamentos: [Glorot e Bengio, 2010](https://proceedings.mlr.press/v9/glorot10a.html) e [He et al., 2015](https://arxiv.org/abs/1502.01852).

Próxima: **Aula 08 — Losses, logits e reduções**. A saída afim deste notebook ainda não aplica uma regra de classificação.